# Building a single-name CDS pricing engine, step by step

This notebook builds the engine in `src/cdsengine` from nothing. Every code cell is preceded by two things:

* **Step** – exactly what the cell does.
* **Why** – the reason the model is built that way.

Each piece is written inline first, in the plainest possible Python, and then checked against the packaged engine with an `assert`. If the notebook runs to the end, the explanation and the production code agree.

**Road map**

| Part | Content |
|---|---|
| 1 | The textbook model: annual coupons, default at period end. Reproduces the course notebook. |
| 2 | Where the textbook model departs from the market |
| 3 | Dates and day counts of a standard CDS |
| 4 | Curves with piecewise-constant rates |
| 5 | Protection leg in closed form |
| 6 | Premium leg, including coupon accrued to default |
| 7 | Par spread, clean and dirty value, upfront |
| 8 | Bootstrapping the hazard curve |
| 9 | Conventional spread, upfront and price |
| 10 | Risk: RPV01, CS01, IR01, Rec01, jump-to-default |
| 11 | Cross-check against QuantLib |

Sources: the *Credit Analytics II* course notebook (WFC and CCMO data) and the *Markit CDS Indices Primer*, section 3 (market conventions and the worked trade example).

**Step.** Import the standard tools and the packaged engine under the alias `eng`.

**Why.** The inline code below uses only `math`, `numpy`, `scipy` and `datetime`. The package is imported solely so that each inline result can be compared with it.

In [1]:
import math
from datetime import date, timedelta

import numpy as np
import pandas as pd
from scipy.optimize import brentq

import cdsengine as eng

pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

# Part 1 — The textbook model

## 1.1 Market data

**Step.** Store the quotes from the course notebook: par spreads in basis points for Wells Fargo (WFC) and Clear Channel (CCMO), and risk-free discount factors.

**Why.** These are the only inputs a CDS model needs besides a recovery assumption. Note that 4Y and 6Y are missing: the market quotes a handful of tenors, and the model has to fill the gaps.

In [2]:
quotes = pd.DataFrame({
    "Maturity": [1, 2, 3, 5, 7],
    "WFC":      [50, 77, 94, 125, 133],
    "CCMO":     [751, 1164, 1874, 4156, 6083],
    "Discount": [0.97, 0.94, 0.92, 0.86, 0.81],
})
quotes

,Maturity,WFC,CCMO,Discount
0,1,50,751,0.970000
1,2,77,1164,0.940000
2,3,94,1874,0.920000
3,5,125,4156,0.860000
4,7,133,6083,0.810000


## 1.2 Filling the gaps in the discount curve

**Step.** Interpolate discount factors linearly in $\ln DF$, anchoring $DF(0)=1$.

**Why.** Write $DF(t)=\exp\!\big(-\int_0^t f(u)\,du\big)$, where $f$ is the instantaneous forward rate. If $\ln DF$ is a straight line between two nodes, its slope $-f$ is constant there. So "log-linear in DF" and "piecewise-constant forward rate" are the same statement. It is the simplest interpolation that keeps discount factors positive and decreasing, and it is the one the ISDA standard model prescribes. Interpolating DF itself linearly would instead imply a forward rate that rises within every interval and jumps at each node.

In [3]:
def interp_df(known_t, known_df, t):
    known_t = np.concatenate([[0.0], known_t])
    known_ln = np.concatenate([[0.0], np.log(known_df)])
    return np.exp(np.interp(t, known_t, known_ln))

t = np.arange(8)
df = interp_df(quotes.Maturity, quotes.Discount, t)

assert np.allclose(df, eng.df_log_linear(quotes.Maturity, quotes.Discount, t))
pd.DataFrame({"t": t, "DF": df, "forward rate over the previous year": np.r_[np.nan, -np.diff(np.log(df))]})

,t,DF,forward rate over the previous year
0,0,1.000000,NaN
1,1,0.970000,0.030459
2,2,0.940000,0.031416
3,3,0.920000,0.021506
4,4,0.889494,0.033721
5,5,0.860000,0.033721
6,6,0.834626,0.029949
7,7,0.810000,0.029949


The forward rate is identical for years 4 and 5, and again for years 6 and 7: one constant per quoted interval, as intended.

## 1.3 Filling the gaps in the spreads

**Step.** Interpolate the spreads linearly to get 4Y and 6Y, and convert basis points to decimals.

**Why.** The textbook bootstrap below solves one survival probability per year, so it needs one spread per year. This is a classroom convenience. The ISDA model in Part 8 does *not* interpolate spreads; it bootstraps only the quoted tenors and lets the piecewise-constant hazard rate fill the gaps, which cannot manufacture an arbitrage the way an interpolated spread can.

In [4]:
wfc = np.interp(t, quotes.Maturity, quotes.WFC) / 1e4
ccmo = np.interp(t, quotes.Maturity, quotes.CCMO) / 1e4
wfc[0] = ccmo[0] = 0.0
pd.DataFrame({"t": t, "WFC (bp)": wfc * 1e4, "CCMO (bp)": ccmo * 1e4})

,t,WFC (bp),CCMO (bp)
0,0,0.000000,0.000000
1,1,50.000000,751.000000
2,2,77.000000,"1,164.000000"
3,3,94.000000,"1,874.000000"
4,4,109.500000,"3,015.000000"
5,5,125.000000,"4,156.000000"
6,6,129.000000,"5,119.500000"
7,7,133.000000,"6,083.000000"


## 1.4 The annual bootstrap

A CDS has two legs. Let $P_n$ be the probability of surviving to year $n$, $S_N$ the $N$-year spread, $L = 1-R$ the loss given default and $\Delta t = 1$.

* Premium leg: the buyer pays the spread while the name is alive, $\;PL_N = S_N \sum_{n=1}^{N} DF_n\,P_n\,\Delta t$
* Protection leg: the seller pays $L$ in the year of default, $\;DL_N = L \sum_{n=1}^{N} DF_n\,(P_{n-1}-P_n)$

A par spread is by definition the spread that makes the contract worth zero, $PL_N = DL_N$. Every $P_n$ with $n<N$ is already known from shorter contracts, so the equation has one unknown, $P_N$:

$$P_N=\frac{\sum_{n=1}^{N-1} DF_n\big[L\,P_{n-1}-(L+\Delta t\,S_N)\,P_n\big] + DF_N\,L\,P_{N-1}}{DF_N\,(L+\Delta t\,S_N)}$$

**Step.** Apply that formula year by year, starting from $P_0 = 1$.

**Why.** Solving shortest maturity first is what "bootstrap" means: each quote pins down exactly one new number, so no simultaneous system is needed. For $N=1$ the formula collapses to $P_1 = L/(L+S_1)$, which is easy to check by hand.

In [5]:
def bootstrap_annual(t, df, spreads, recovery):
    L = 1.0 - recovery
    P = [1.0]
    for N in range(1, len(t)):
        S = spreads[N]
        known = sum(df[n] * (L * P[n - 1] - (L + S) * P[n]) for n in range(1, N))
        P.append((known + df[N] * L * P[N - 1]) / (df[N] * (L + S)))
    return np.array(P)

P_wfc = bootstrap_annual(t, df, wfc, recovery=0.50)

assert math.isclose(P_wfc[1], 0.5 / (0.5 + 0.0050))            # hand check: 0.990099
assert np.allclose(P_wfc, eng.bootstrap_simple(t, df, wfc, 0.50))
P_wfc

array([1.        , 0.99009901, 0.96964924, 0.94496611, 0.91536605,
       0.88053648, 0.8543602 , 0.82738795])

These are the course notebook's WFC numbers (99.0099%, 96.9649%, 94.4966%, ...).

## 1.5 From survival probabilities to hazard rates

**Step.** Compute $\lambda_m = -\ln(P_m/P_{m-1})/\Delta t$ for each year.

**Why.** The hazard rate is the default intensity: over a short interval, the probability of defaulting given survival so far is $\lambda\,dt$. With a constant $\lambda$ on $(t_{m-1},t_m]$, survival decays as $P_m = P_{m-1}e^{-\lambda_m\Delta t}$; inverting gives the formula. Hazard rates are the natural language for a credit curve because the only constraint is $\lambda \ge 0$, which is far easier to police than "survival probabilities must be decreasing and positive".

In [6]:
def hazard_rates(P):
    return np.r_[0.0, -np.log(P[1:] / P[:-1])]

lam_wfc = hazard_rates(P_wfc)
pd.DataFrame({"t": t, "Survival": P_wfc, "Marginal PD": np.r_[0, -np.diff(P_wfc)], "Hazard": lam_wfc})

,t,Survival,Marginal PD,Hazard
0,0,1.000000,0.000000,0.000000
1,1,0.990099,0.009901,0.009950
2,2,0.969649,0.020450,0.020871
3,3,0.944966,0.024683,0.025785
4,4,0.915366,0.029600,0.031825
5,5,0.880536,0.034830,0.038793
6,6,0.854360,0.026176,0.030178
7,7,0.827388,0.026972,0.032079


## 1.6 The credit triangle

**Step.** Compare the 1Y hazard rate with the shortcut $\lambda \approx S/(1-R)$, then do the same at longer tenors.

**Why.** Per unit of time the buyer pays $S$ and expects to receive $\lambda(1-R)$; equating them gives the triangle. It is exact only when the hazard rate is flat and discounting is ignored, so it works at 1Y and drifts as soon as the curve has a slope. It is a sanity check, not a pricing formula.

In [7]:
pd.DataFrame({"t": t[1:], "bootstrapped hazard": lam_wfc[1:], "S / (1-R)": wfc[1:] / 0.5,
              "error": wfc[1:] / 0.5 - lam_wfc[1:]})

,t,bootstrapped hazard,S / (1-R),error
0,1,0.009950,0.010000,0.000050
1,2,0.020871,0.015400,-0.005471
2,3,0.025785,0.018800,-0.006985
3,4,0.031825,0.021900,-0.009925
4,5,0.038793,0.025000,-0.013793
5,6,0.030178,0.025800,-0.004378
6,7,0.032079,0.026600,-0.005479


## 1.7 Sensitivity to the recovery assumption

**Step.** Re-run the bootstrap for several recovery rates with the spreads held fixed.

**Why.** The spread only pins down the product $\lambda(1-R)$. Raise $R$ and the same spread must be explained by a higher default probability. Recovery is not observable before default, so this table is the size of the model risk in any survival probability quoted from a CDS curve.

In [8]:
pd.DataFrame({f"R={r:.0%}": bootstrap_annual(t, df, wfc, r) for r in (0.10, 0.30, 0.50, 0.70, 0.90)}, index=t)

,R=10%,R=30%,R=50%,R=70%,R=90%
0,1.000000,1.000000,1.000000,1.000000,1.000000
1,0.994475,0.992908,0.990099,0.983607,0.952381
2,0.982987,0.978196,0.969649,0.950086,0.859653
3,0.968988,0.960330,0.944966,0.910189,0.756536
4,0.952023,0.938759,0.915366,0.863093,0.643739
5,0.931817,0.913180,0.880536,0.808658,0.524143
6,0.916325,0.893707,0.854360,0.768967,0.449603
7,0.900173,0.873490,0.827388,0.728774,0.380147


## 1.8 CCMO: when the formula returns nonsense

**Step.** Run the same bootstrap on the CCMO spreads with 10% recovery.

**Why.** This is the most instructive case in the course notebook. Nothing in the algebra stops $P_N$ from going negative, so the formula has to be watched.

In [9]:
P_ccmo = bootstrap_annual(t, df, ccmo, recovery=0.10)
pd.DataFrame({"t": t, "Spread (bp)": ccmo * 1e4, "Survival": P_ccmo})

,t,Spread (bp),Survival
0,0,0.000000,1.000000
1,1,751.000000,0.922982
2,2,"1,164.000000",0.778580
3,3,"1,874.000000",0.528920
4,4,"3,015.000000",0.170524
5,5,"4,156.000000",-0.111808
6,6,"5,119.500000",-0.248629
7,7,"6,083.000000",-0.303070


Survival is $-11\%$ at 5Y. The reason is economic, not numerical. A 5Y spread of 4156bp means paying about 42% of notional a year. The most protection can ever pay is $1-R = 90\%$, once. After the 4Y point there is only 17% survival left, and even if all of it defaulted in year 5 the protection leg could not catch up with a premium leg priced at 4156bp. No probability distribution reprices these quotes.

What went wrong is the input, in three ways: names this distressed trade on an upfront basis rather than as running par spreads; the 4Y point was invented by linear interpolation between 1874 and 4156; and 10% recovery is an assumption. An engine must refuse such inputs rather than hand back NaN hazard rates.

**Step.** Call the packaged version, which validates each $P_N$.

**Why.** A probability must lie in $(0, P_{N-1}]$. Checking that at every step turns a silent wrong answer into an explicit error at the first offending quote.

In [10]:
try:
    eng.bootstrap_simple(t, df, ccmo, 0.10)
except eng.CalibrationError as err:
    print("CalibrationError:", err)

CalibrationError: t=5: implied survival -0.1118 after 0.1705 is not a valid probability; these spreads cannot be par spreads at this recovery


# Part 2 — Where the textbook model departs from the market

| Textbook model | Standard contract (Markit primer, ISDA standard model) | Consequence |
|---|---|---|
| Coupons once a year | Quarterly, on the 20th of Mar/Jun/Sep/Dec | Premium leg timing |
| $\Delta t = 1$ | Actual/360 | A year of coupons costs 365/360 of the spread |
| Default paid at year end | Paid when it happens | Protection is worth more |
| No coupon for the broken period | Coupon accrued to the default date is owed | Premium leg is worth more |
| Spread paid as quoted | Fixed 100bp or 500bp coupon plus an upfront amount | Needs a spread-to-upfront conversion |
| Contract starts today, no accrual | Full first coupon; buyer is rebated the accrued | Clean vs dirty value |

Parts 3 to 9 remove these simplifications one at a time.

# Part 3 — Dates and day counts

## 3.1 Two day counts

**Step.** Define Act/360 and Act/365F.

**Why.** They do different jobs and must not be mixed. Act/360 is contractual: it determines the coupon cash actually paid. Act/365F is a modelling choice: it is the clock on which the curves measure $t$ in $e^{-\lambda t}$. Using the wrong one in either place is the most common source of small, persistent pricing breaks.

In [11]:
def act360(d0, d1): return (d1 - d0).days / 360
def act365(d0, d1): return (d1 - d0).days / 365

## 3.2 Business days

**Step.** Treat Saturday and Sunday as the only holidays and roll a bad day forward ("Following").

**Why.** The standard model deliberately uses a weekends-only calendar for the coupon schedule so that every participant generates the same dates without exchanging holiday files.

In [12]:
ONE_DAY = timedelta(days=1)

def following(d):
    while d.weekday() >= 5:
        d += ONE_DAY
    return d

def add_months(d, n):
    y, m = divmod(d.year * 12 + d.month - 1 + n, 12)
    return date(y, m + 1, d.day)          # only ever called on the 20th

## 3.3 Coupon dates and the standard maturity

**Step.** Find the latest 20 Mar/Jun/Sep/Dec on or before a date, then derive the standard maturity for a tenor.

**Why.** Fixing coupon dates makes contracts fungible: a 5Y CDS traded today and one traded next week share every cash-flow date, so they net exactly, which is what central clearing requires. Since 2015 the on-the-run contract rolls only twice a year (20 March and 20 September), so every standard maturity is a 20 June or a 20 December.

In [13]:
def previous_imm20(d):
    c = date(d.year, 12, 20)
    while c > d:
        c = add_months(c, -3)
    return c

def standard_maturity(trade_date, tenor_months):
    anchor = previous_imm20(trade_date)
    if anchor.month in (6, 12):           # between a quarterly date and the next roll
        anchor = add_months(anchor, -3)
    return add_months(anchor, tenor_months + 3)

VAL = date(2020, 12, 14)                  # the course notebook's valuation date
assert standard_maturity(VAL, 60) == eng.cds_maturity(VAL, 60) == date(2025, 12, 20)
[standard_maturity(VAL, 12 * k) for k in (1, 2, 3, 5, 7)]

[datetime.date(2021, 12, 20),
 datetime.date(2022, 12, 20),
 datetime.date(2023, 12, 20),
 datetime.date(2025, 12, 20),
 datetime.date(2027, 12, 20)]

## 3.4 The coupon schedule

**Step.** Build the remaining coupon periods of a contract as seen from the valuation date $T$:

1. step-in date $= T+1$ calendar day;
2. the first period starts at the coupon date on or before the step-in date;
3. step quarterly to maturity;
4. roll every date Following, except the maturity;
5. let the last period accrue one extra day.

**Why.**
* *Step-in $T+1$*: a trade done today gives protection from the start of tomorrow, so risk and accrual both begin then.
* *Full first coupon*: whenever you trade, you pay the whole current coupon on the next date and are rebated the accrued part up front. All contracts on a name therefore pay the same coupon on the same day.
* *Maturity unadjusted, plus one day*: protection covers the maturity date itself, so the buyer pays for that day too.

In [14]:
def schedule(val, maturity):
    stepin = val + ONE_DAY
    first = previous_imm20(stepin)
    if following(first) > stepin:         # the 20th fell on a weekend just before step-in
        first = add_months(first, -3)
    unadjusted = [first]
    while unadjusted[-1] < maturity:
        unadjusted.append(add_months(unadjusted[-1], 3))
    adjusted = [following(d) for d in unadjusted]
    periods = []
    for i in range(len(unadjusted) - 1):
        is_last = i == len(unadjusted) - 2
        accrual_end = maturity + ONE_DAY if is_last else adjusted[i + 1]
        periods.append((adjusted[i], accrual_end, adjusted[i + 1]))   # (start, end, pay)
    return periods

sched = schedule(VAL, date(2025, 12, 20))
assert sched == [(p.accrual_start, p.accrual_end, p.pay_date) for p in eng.cds_schedule(VAL, date(2025, 12, 20))]
pd.DataFrame(sched, columns=["accrual start", "accrual end", "pay date"]).assign(
    days=lambda x: [(e - s).days for s, e in zip(x["accrual start"], x["accrual end"])]).iloc[[0, 1, 2, -2, -1]]

,accrual start,accrual end,pay date,days
0,2020-09-21,2020-12-21,2020-12-21,91
1,2020-12-21,2021-03-22,2021-03-22,91
2,2021-03-22,2021-06-21,2021-06-21,91
19,2025-06-20,2025-09-22,2025-09-22,94
20,2025-09-22,2025-12-21,2025-12-22,90


The first period starts on 21 September 2020 because the 20th was a Sunday. The last period ends on 21 December 2025 (maturity plus one day) but pays on Monday the 22nd because the 20th is a Saturday.

## 3.5 Checking the dates against the Markit primer

**Step.** Reproduce the primer's trade example: CDX.NA.HY.S35, USD 10m, 500bp coupon, bought on 10 November 2020.

**Why.** A worked example from a market data vendor is an independent test of the schedule logic. The primer's figures are an accrual start of 21 September 2020, a first coupon of 91 days worth 126,388.89, and 50 days of accrued worth 69,444.44.

In [15]:
start, end, pay = schedule(date(2020, 11, 10), date(2025, 12, 20))[0]
print("accrual start      :", start)
print("first coupon       :", (end - start).days, "days ->", round(10_000_000 * 0.05 * act360(start, end), 2))
print("accrued, trade date:", (date(2020, 11, 10) - start).days, "days ->", round(10_000_000 * 0.05 * 50 / 360, 2))
print("accrued, step-in   :", (date(2020, 11, 11) - start).days, "days ->", round(10_000_000 * 0.05 * 51 / 360, 2))

accrual start      : 2020-09-21
first coupon       : 91 days -> 126388.89
accrued, trade date: 50 days -> 69444.44
accrued, step-in   : 51 days -> 70833.33


The dates and the coupon match. One difference is worth knowing: the primer counts accrued to the trade date (50 days), whereas the standard model counts to the step-in date (51 days), because the buyer is protected, and so pays, from $T+1$. This engine follows the standard model.

(Two figures in the primer's example are typos and should not be used as test targets: the unwind net cash flow should be $-960{,}000 - 68{,}055.56 = -1{,}028{,}055.56$, and the Whiting payout is $92{,}847.22$.)

# Part 4 — Curves with piecewise-constant rates

**Step.** Represent a curve by the value of $-\ln(\cdot)$ at its node dates and interpolate that linearly in Act/365F time, extending the last slope beyond the final node. One helper serves both curves:

* discount curve: nodes hold $-\ln DF$, the slope is the forward rate $f$;
* credit curve: nodes hold $-\ln Q$, the slope is the hazard rate $\lambda$.

**Why.** This is where "piecewise constant" enters the model, and it does so twice, for two different reasons.

* *Hazard rate.* $N$ quotes can determine $N$ numbers. One constant hazard per interval between quoted maturities gives exactly one unknown per quote, so the bootstrap is well posed and needs no smoothing assumptions.
* *Forward rate.* As in 1.2, it is the simplest arbitrage-free interpolation of discount factors.
* *Together.* Between two consecutive nodes of either curve, $DF(t)\,Q(t)$ is a single exponential $e^{-(f+\lambda)t}$. Integrals of an exponential have closed forms, which is what Parts 5 and 6 exploit.

In [16]:
def make_curve(base, node_dates, neg_logs):
    """Return a function d -> ln(value at d); value is DF or Q."""
    times = [act365(base, d) for d in node_dates]
    def log_value(d):
        x = act365(base, d)
        if x <= 0:
            return 0.0
        if x >= times[-1]:                                   # flat extrapolation
            prev_t, prev_y = (times[-2], neg_logs[-2]) if len(times) > 1 else (0.0, 0.0)
            slope = (neg_logs[-1] - prev_y) / (times[-1] - prev_t)
            return -(neg_logs[-1] + slope * (x - times[-1]))
        return -float(np.interp(x, [0.0, *times], [0.0, *neg_logs]))
    return log_value

def discount_curve(base, node_dates, dfs):
    return make_curve(base, node_dates, [-math.log(x) for x in dfs])

def credit_curve(base, node_dates, hazards):
    times = [0.0] + [act365(base, d) for d in node_dates]
    cumulative = np.cumsum([h * (times[i + 1] - times[i]) for i, h in enumerate(hazards)])
    return make_curve(base, node_dates, list(cumulative))

**Step.** Build the discount curve from the course data, dating each tenor from the valuation date, and compare with the packaged class.

**Why.** From here on everything is in calendar dates, so the year-count tenors of Part 1 have to be placed on the calendar.

In [17]:
DF_DATES = [date(VAL.year + k, VAL.month, VAL.day) for k in (1, 2, 3, 5, 7)]
DF_VALUES = [0.97, 0.94, 0.92, 0.86, 0.81]

lnP = discount_curve(VAL, DF_DATES, DF_VALUES)
DISC = eng.DiscountCurve(VAL, DF_DATES, DF_VALUES)

probe = [date(2021, 6, 20), date(2024, 12, 14), date(2025, 12, 20), date(2029, 1, 1)]
assert all(math.isclose(lnP(d), DISC.log_df(d), abs_tol=1e-15) for d in probe)
pd.DataFrame({"date": probe, "DF": [math.exp(lnP(d)) for d in probe]})

,date,DF
0,2021-06-20,0.984434
1,2024-12-14,0.889453
2,2025-12-20,0.859577
3,2029-01-01,0.784876


# Part 5 — The protection leg

The seller pays $1-R$ at the moment of default. The probability that default happens in $[t, t+dt]$ is $-dQ(t)$, so

$$\text{Protection} = (1-R)\int_{0}^{T} DF(t)\,\big(-dQ(t)\big)$$

Cut $[0,T]$ at every node of both curves. On one segment $[t_0,t_1]$, with $B = DF\cdot Q$, define the two dimensionless numbers

$$\hat f = \ln\frac{DF_0}{DF_1},\qquad \hat h = \ln\frac{Q_0}{Q_1}.$$

Because both rates are constant there, $B(t) = B_0\,e^{-(\hat f+\hat h)u}$ with $u\in[0,1]$, and $-dQ = \hat h\,Q\,du$. The integral is elementary:

$$\int_{t_0}^{t_1} DF\,(-dQ) = \frac{\hat h}{\hat f+\hat h}\,\big(B_0 - B_1\big)$$

Read it as: $B_0-B_1$ is the total loss of "risky discounted value" over the segment, and $\hat h/(\hat f+\hat h)$ is the share of that loss caused by default rather than by discounting.

## 5.1 One segment

**Step.** Code the segment formula, switching to a Taylor series when $\hat f+\hat h$ is tiny.

**Why.** The closed form is $0/0$ when $\hat f+\hat h\to 0$ (for instance a one-day segment, or zero rates). Expanding $\frac{1-e^{-x}}{x} = 1-\frac{x}{2}+\frac{x^2}{6}-\dots$ removes the division and keeps full precision.

In [18]:
def default_pv_segment(lnP0, lnQ0, lnP1, lnQ1):
    h = lnQ0 - lnQ1
    x = (lnP0 - lnP1) + h                      # f-hat + h-hat
    B0 = math.exp(lnP0 + lnQ0)
    if abs(x) < 1e-4:
        return B0 * h * (1 - x / 2 + x**2 / 6 - x**3 / 24 + x**4 / 120)
    return h / x * (B0 - math.exp(lnP1 + lnQ1))

## 5.2 The whole leg

**Step.** Collect the grid (valuation date, every curve node strictly before maturity, maturity), sum the segment values, and multiply by $1-R$.

**Why.** The grid must contain every date at which either rate changes; otherwise a segment would hide a kink and the single-exponential assumption would be false. Protection starts at the valuation date because a default from tomorrow onward is covered and time is measured in whole days.

In [19]:
def protection_leg(val, maturity, recovery, lnP, lnQ, node_dates):
    grid = [val] + sorted(d for d in set(node_dates) if val < d < maturity) + [maturity]
    total = sum(default_pv_segment(lnP(a), lnQ(a), lnP(b), lnQ(b)) for a, b in zip(grid, grid[1:]))
    return (1 - recovery) * total

**Step.** Price 5Y protection on an illustrative credit curve and compare three ways of doing the integral: the closed form, a brute-force sum over every single day, and a midpoint rule with one step per coupon period.

**Why.** The daily sum is slow but assumption-free, so it validates the closed form. The midpoint rule is what the course notebook's `MidPointCdsEngine` does, so the third number shows the size of the approximation the ISDA model removes.

In [20]:
MATS = [standard_maturity(VAL, 12 * k) for k in (1, 2, 3, 5, 7)]
demo_hazards = [0.010, 0.021, 0.026, 0.036, 0.032]
lnQ = credit_curve(VAL, MATS, demo_hazards)
CRED = eng.CreditCurve(VAL, MATS, demo_hazards)
NODES = DF_DATES + MATS
T5 = MATS[3]

closed = protection_leg(VAL, T5, 0.40, lnP, lnQ, NODES)

daily, d = 0.0, VAL
while d < T5:
    nxt = d + ONE_DAY
    daily += math.exp(0.5 * (lnP(d) + lnP(nxt))) * (math.exp(lnQ(d)) - math.exp(lnQ(nxt)))
    d = nxt
daily *= 0.6

midpoint, prev = 0.0, VAL
for _, _, pay in schedule(VAL, T5):
    mid = prev + (pay - prev) / 2
    midpoint += math.exp(lnP(mid)) * (math.exp(lnQ(prev)) - math.exp(lnQ(min(pay, T5))))
    prev = min(pay, T5)
midpoint *= 0.6

assert math.isclose(closed, eng.protection_leg_pv(VAL, T5, 0.40, DISC, CRED), abs_tol=1e-15)
assert abs(closed - daily) < 1e-8
pd.DataFrame({"protection PV": [closed, daily, midpoint],
              "difference in USD per 10m": [0.0, (daily - closed) * 1e7, (midpoint - closed) * 1e7]},
             index=["closed form", "daily sum", "midpoint per coupon period"])

,protection PV,difference in USD per 10m
closed form,0.066696,0.000000
daily sum,0.066696,-0.000572
midpoint per coupon period,0.066696,4.833409


The closed form and the daily sum agree to a fraction of a cent on USD 10m. The midpoint rule is off by about USD 5: small in money terms, but thousands of times larger than the closed form's error, and it grows with the spread and with the length of the coupon period. It is a genuine approximation in the course notebook's engine, not a bug.

# Part 6 — The premium leg

Per unit of coupon the buyer pays two things.

1. **Scheduled coupons**, each paid only if the name survives its period: $\;\sum_i \Delta_i\,DF(\text{pay}_i)\,Q(\text{end}_i)$, with $\Delta_i$ in Act/360.
2. **Accrual on default**: if the name defaults inside a period, the buyer owes the coupon accrued from the period start $s_i$ to the default date: $\;\sum_i\int (t-s_i)\,DF(t)\,(-dQ(t))$.

Their sum is the **risky annuity**, or **RPV01**: the present value of paying 1 unit of coupon per year until default or maturity. A coupon $c$ is worth $c\times$RPV01.

## 6.1 Scheduled coupons

**Step.** For each period multiply the Act/360 fraction, the discount factor at the payment date and the survival probability one day before the accrual end.

**Why.** Discounting uses the payment date because that is when cash moves. Survival is observed at the last day of the accrual period, i.e. accrual end minus one: the coupon is earned if the name is still alive at the close of that day. In the final period that day is the maturity date itself.

In [21]:
def coupon_annuity(val, maturity, lnP, lnQ):
    return sum(act360(start, end) * math.exp(lnP(pay) + lnQ(end - ONE_DAY))
               for start, end, pay in schedule(val, maturity))

## 6.2 Accrual on default

On a segment with constant rates the integrand is $(t-s)\times$ an exponential, which integrates by parts to

$$\int_{t_0}^{t_1}(t-s)\,DF\,(-dQ)=\frac{\hat h}{\hat f+\hat h}\left[(t_1-t_0)\left(\frac{B_0-B_1}{\hat f+\hat h}-B_1\right)+(t_0-s)\,(B_0-B_1)\right]$$

**Step.** Code the segment formula with its Taylor fallback, then sum over every coupon period, splitting each period at any curve node inside it. Convert the result from Act/365F years to Act/360 with the factor 365/360.

**Why.**
* Ignoring this term understates the premium leg, most of all for wide spreads where default inside a period is likely.
* A simpler approximation, "half a coupon on default", is accurate only when the hazard rate is low; the integral is exact for the model.
* Splitting at interior nodes matters because the hazard rate can change in the middle of a coupon period. The original ISDA code treated each period as one flat segment and added a half-day bias; later versions corrected both. The packaged engine uses the corrected form by default and keeps the original behind `legacy_accrual=True` for reconciling with older systems.
* The $-1$ day offsets mirror 6.1: the accrual clock starts at the close of the day before the period begins.

In [22]:
def accrual_segment(t0, t1, ts, lnP0, lnQ0, lnP1, lnQ1):
    h = lnQ0 - lnQ1
    x = (lnP0 - lnP1) + h
    B0 = math.exp(lnP0 + lnQ0)
    if abs(x) < 1e-4:
        return h * B0 * ((t0 - ts) * (1 - x / 2 + x**2 / 6 - x**3 / 24)
                         + (t1 - t0) * (1 / 2 - x / 3 + x**2 / 8 - x**3 / 30))
    B1 = math.exp(lnP1 + lnQ1)
    return h / x * ((t1 - t0) * ((B0 - B1) / x - B1) + (t0 - ts) * (B0 - B1))

def default_accrual_annuity(val, maturity, lnP, lnQ, node_dates):
    stepin, total = val + ONE_DAY, 0.0
    for start, end, _ in schedule(val, maturity):
        a, b = max(start, stepin) - ONE_DAY, end - ONE_DAY
        ts = act365(val, start - ONE_DAY)
        grid = [a] + sorted(d for d in set(node_dates) if a < d < b) + [b]
        for d0, d1 in zip(grid, grid[1:]):
            total += accrual_segment(act365(val, d0), act365(val, d1), ts,
                                     lnP(d0), lnQ(d0), lnP(d1), lnQ(d1))
    return total * 365 / 360

ann = coupon_annuity(VAL, T5, lnP, lnQ)
aod = default_accrual_annuity(VAL, T5, lnP, lnQ, NODES)

pkg_ann, pkg_aod, pkg_accrued = eng.premium_leg_annuities(VAL, T5, DISC, CRED)
assert math.isclose(ann, pkg_ann, abs_tol=1e-14) and math.isclose(aod, pkg_aod, abs_tol=1e-14)
print(f"scheduled coupons  : {ann:.6f}")
print(f"accrual on default : {aod:.6f}   ({aod / (ann + aod):.2%} of the annuity)")
print(f"RPV01 (dirty)      : {ann + aod:.6f}")

scheduled coupons  : 4.710012
accrual on default : 0.014099   (0.30% of the annuity)
RPV01 (dirty)      : 4.724110


# Part 7 — Par spread, clean and dirty value, upfront

**Step.** Combine the legs into the numbers a trader quotes, for a 5Y contract with the standard 100bp coupon:

* accrued $= $ Act/360 from the first accrual start to the step-in date;
* dirty value $=$ protection $- c\times$ RPV01(dirty);
* clean RPV01 $=$ dirty RPV01 $-$ accrued $\times DF(\text{settle})$;
* clean value $=$ protection $- c\times$ RPV01(clean);
* par spread $=$ protection / RPV01(clean);
* upfront $=$ clean value $/\,DF(\text{settle})$, with settlement at $T+3$ business days.

**Why.**
* *Dirty vs clean.* The dirty value is the true economic value: the buyer really will pay the full next coupon. But part of that coupon covers a period before the trade, so the seller hands the accrued amount back at settlement. Quoting clean removes the saw-tooth that the coupon calendar would otherwise put into prices. It is the same idea as clean and dirty bond prices.
* *Why accrued is multiplied by DF(settle).* The rebate is paid on the settlement date, three business days out, so today it is worth slightly less than its face amount.
* *Par spread on the clean annuity.* The par spread is the coupon that makes the clean value zero, which gives the identity clean value $=(S_{par}-c)\times$ RPV01(clean). That is the primer's $UF \approx (\text{coupon} - \text{par spread})\times RPV01$, seen from the buyer's side.
* *Dividing by DF(settle).* Model values are as of today; the upfront is a cash amount paid at settlement.

In [23]:
def value(val, maturity, coupon, recovery, lnP, lnQ, node_dates):
    stepin = val + ONE_DAY
    settle = val
    for _ in range(3):                                  # T+3 business days
        settle = following(settle + ONE_DAY)
    df_settle = math.exp(lnP(settle))
    prot = protection_leg(val, maturity, recovery, lnP, lnQ, node_dates)
    rpv01_dirty = coupon_annuity(val, maturity, lnP, lnQ) + default_accrual_annuity(val, maturity, lnP, lnQ, node_dates)
    accrued = act360(schedule(val, maturity)[0][0], stepin)
    rpv01_clean = rpv01_dirty - accrued * df_settle
    return {"protection_pv": prot, "rpv01_dirty": rpv01_dirty, "accrued": accrued, "rpv01_clean": rpv01_clean,
            "pv_dirty": prot - coupon * rpv01_dirty, "pv_clean": prot - coupon * rpv01_clean,
            "par_spread": prot / rpv01_clean, "upfront": (prot - coupon * rpv01_clean) / df_settle}

v = value(VAL, T5, 0.01, 0.40, lnP, lnQ, NODES)
pv = eng.value_cds(eng.CDS(T5, 0.01, 0.40), VAL, DISC, CRED)
for key in v:
    assert math.isclose(v[key], getattr(pv, key), abs_tol=1e-14), key
assert math.isclose(v["pv_clean"], (v["par_spread"] - 0.01) * v["rpv01_clean"])
pd.Series(v)

protection_pv   0.066696
rpv01_dirty     4.724110
accrued         0.236111
rpv01_clean     4.488058
pv_dirty        0.019454
pv_clean        0.021815
par_spread      0.014861
upfront         0.021820
dtype: float64

The par spread is above the 100bp coupon, so the contract is an asset to the protection buyer, who therefore pays an upfront amount to enter it.

# Part 8 — Bootstrapping the hazard curve

So far the hazard rates were made up. Now they are implied from quotes.

**Step.** For each quoted maturity, shortest first:

1. keep every hazard rate already found;
2. treat the hazard on the newest interval as the unknown $h$;
3. define $g(h)$ = clean value of a CDS that pays the quoted spread as its coupon;
4. find the root $g(h)=0$ with Brent's method;
5. fail loudly if no root exists.

**Why.**
* *Same logic as Part 1, different solver.* One quote, one unknown. The textbook leg formulas were linear in $P_N$ so algebra sufficed; the ISDA legs are exponential in $h$, so a one-dimensional root finder replaces the algebra.
* *Why clean value zero.* That is the definition of a par spread under the standard contract terms.
* *Why Brent.* $g$ is continuous and increasing in $h$ (more default risk is good for the protection buyer), so a bracketing method converges reliably without derivatives.
* *The two failure modes.* If $g(0)>0$, the quote is so low relative to shorter maturities that it needs a negative hazard. If $g$ never turns positive however large $h$ becomes, even immediate default cannot justify the spread, which is the CCMO situation. Both are arbitrage signals in the inputs, and both raise an error rather than return a curve.

In [24]:
def bootstrap(val, maturities, spreads, recovery, lnP, df_nodes):
    hazards = []
    for k, (mat, s) in enumerate(zip(maturities, spreads)):
        def g(h):
            trial = credit_curve(val, maturities[:k + 1], hazards + [h])
            return value(val, mat, s, recovery, lnP, trial, df_nodes + maturities[:k + 1])["pv_clean"]
        if g(0.0) > 0:
            raise ValueError(f"{mat}: implied hazard is negative")
        hi = 1.0
        while g(hi) < 0:
            hi *= 2
            if hi > 50:
                raise ValueError(f"{mat}: no hazard rate reprices {s * 1e4:.0f}bp")
        hazards.append(brentq(g, 0.0, hi, xtol=1e-14))
    return hazards

WFC = [0.0050, 0.0077, 0.0094, 0.0125, 0.0133]
wfc_hazards = bootstrap(VAL, MATS, WFC, 0.50, lnP, DF_DATES)
WFC_CURVE = eng.bootstrap_credit_curve(VAL, MATS, WFC, 0.50, DISC)
assert np.allclose(wfc_hazards, WFC_CURVE.hazards, atol=1e-12)

lnQ_wfc = credit_curve(VAL, MATS, wfc_hazards)
pd.DataFrame({"maturity": MATS, "spread (bp)": np.array(WFC) * 1e4, "hazard": wfc_hazards,
              "survival": [math.exp(lnQ_wfc(m)) for m in MATS],
              "repriced par spread (bp)": [value(VAL, m, s, 0.5, lnP, lnQ_wfc, NODES)["par_spread"] * 1e4
                                           for m, s in zip(MATS, WFC)]})

,maturity,spread (bp),hazard,survival,repriced par spread (bp)
0,2021-12-20,50.000000,0.010099,0.989787,50.000000
1,2022-12-20,77.000000,0.021362,0.968867,77.000000
2,2023-12-20,94.000000,0.026465,0.943563,94.000000
3,2025-12-20,125.000000,0.036129,0.877702,125.000000
4,2027-12-20,133.000000,0.031924,0.823414,133.000000


Every quote reprices exactly, and only the five quoted tenors were used: there is no invented 4Y or 6Y point.

**Step.** Compare the ISDA hazard rates with the textbook ones from Part 1.

**Why.** It quantifies what Part 2's simplifications were worth. The textbook figures for years 4-5 and 6-7 are averaged because that model produced one hazard per year.

In [25]:
textbook = [lam_wfc[1], lam_wfc[2], lam_wfc[3], lam_wfc[4:6].mean(), lam_wfc[6:8].mean()]
pd.DataFrame({"interval ending": ["1Y", "2Y", "3Y", "5Y", "7Y"], "textbook": textbook, "ISDA": wfc_hazards,
              "difference": np.array(wfc_hazards) - np.array(textbook)})

,interval ending,textbook,ISDA,difference
0,1Y,0.009950,0.010099,0.000149
1,2Y,0.020871,0.021362,0.000491
2,3Y,0.025785,0.026465,0.000680
3,5Y,0.035309,0.036129,0.000820
4,7Y,0.031129,0.031924,0.000795


**Step.** Feed the CCMO quotes to the ISDA bootstrap.

**Why.** To confirm that a better model does not rescue impossible inputs; it only reports them properly.

In [26]:
CCMO = [0.0751, 0.1164, 0.1874, 0.4156, 0.6083]
try:
    eng.bootstrap_credit_curve(VAL, MATS, CCMO, 0.10, DISC)
except eng.CalibrationError as err:
    print("CalibrationError:", err)

CalibrationError: quote 3 (2025-12-20, 4156.0bp): no hazard rate reprices the quote; even immediate default after the previous node is not enough protection value for this spread


# Part 9 — Conventional spread, upfront and price

A standard contract pays a fixed coupon (100bp or 500bp), so what is negotiated is the upfront amount. Dealers nevertheless prefer to talk in spreads. The **conventional spread** is the agreed translation.

**Step.** Convert a conventional spread to an upfront amount:

1. build a *flat* hazard curve from that single spread, by bootstrapping one quote at the contract's own maturity;
2. value the fixed-coupon contract on that flat curve;
3. report the clean upfront. Price per 100 is $100\times(1-\text{upfront})$.

The reverse direction is a root search over the spread.

**Why.**
* *Why a flat curve.* Both counterparties must compute the same cash amount from the same quoted number. A full curve would make the result depend on each side's other quotes; a single-point flat curve depends on nothing else. The conventional spread is therefore a quoting device, not a par spread: the two coincide only when the true curve happens to be flat. The primer makes the same distinction between RPV01 and RPV01*.
* *Why price = 100 − points upfront.* It lets a CDS be quoted like a bond: a price above par means the buyer of protection receives money, as in the primer's example below.

In [27]:
def upfront_from_spread(val, maturity, coupon, recovery, spread, lnP, df_nodes):
    flat_hazard = bootstrap(val, [maturity], [spread], recovery, lnP, df_nodes)
    flat = credit_curve(val, [maturity], flat_hazard)
    return value(val, maturity, coupon, recovery, lnP, flat, df_nodes + [maturity])["upfront"]

u = upfront_from_spread(VAL, T5, 0.01, 0.40, 0.0125, lnP, DF_DATES)
assert math.isclose(u, eng.upfront_from_conventional_spread(eng.CDS(T5, 0.01, 0.40), VAL, 0.0125, DISC), abs_tol=1e-12)
back = eng.conventional_spread_from_upfront(eng.CDS(T5, 0.01, 0.40), VAL, u, DISC)
print(f"125bp conventional spread, 100bp coupon -> upfront {u:.4%} of notional, price {100 * (1 - u):.3f}")
print(f"and back again -> {back * 1e4:.4f}bp")

125bp conventional spread, 100bp coupon -> upfront 1.1197% of notional, price 98.880
and back again -> 125.0000bp


**Step.** Apply the conversion to the two quotes in the Markit primer's trade example (CDX.NA.HY.S35 5Y, 500bp coupon): 332.80bp at a price of 107.61 on 10 November 2020, and 283.37bp at 109.60 on 8 February 2021.

**Why.** It is an external check on the whole chain: dates, both legs, accrued and the flat-curve convention. Two assumptions are needed that the primer does not state, so treat this as a sanity check on conventions rather than a precision test: the index is priced as a single name with the 30% recovery standard for North American high yield, and the swap curve is replaced by a flat 0.4% rate, roughly the level of 5Y USD swaps at the time.

In [28]:
rows = []
for val, spread, quoted in [(date(2020, 11, 10), 0.033280, 107.61), (date(2021, 2, 8), 0.028337, 109.60)]:
    cds = eng.CDS(date(2025, 12, 20), 0.05, 0.30, notional=10_000_000)
    flat_rates = eng.DiscountCurve.flat(val, 0.004)
    up = eng.upfront_from_conventional_spread(cds, val, spread, flat_rates)
    rows.append({"trade date": val, "spread (bp)": spread * 1e4, "model price": eng.clean_price(up),
                 "primer price": quoted, "model upfront (USD)": up * cds.notional,
                 "primer upfront (USD)": (100 - quoted) / 100 * cds.notional})
pd.DataFrame(rows)

,trade date,spread (bp),model price,primer price,model upfront (USD),primer upfront (USD)
0,2020-11-10,332.800000,107.603220,107.610000,"-760,322.027715","-761,000.000000"
1,2021-02-08,283.370000,109.592830,109.600000,"-959,282.981882","-960,000.000000"


Both prices land within a cent per 100 of the published ones. The upfront is negative: the spread is below the 500bp coupon, so the protection buyer is overpaying in coupon and is compensated up front, exactly as the primer describes.

# Part 10 — Risk

**Step.** Compute five risk measures for a long-protection 5Y WFC position (100bp coupon, 50% recovery) by bump and reprice:

| Measure | Bump | What is held fixed |
|---|---|---|
| RPV01 | none; it is the clean risky annuity | |
| CS01 | +1bp on every par spread, re-bootstrap | rates, recovery |
| Bucketed CS01 | +1bp on one tenor at a time | rates, recovery |
| IR01 | +1bp on every zero rate, re-bootstrap | spreads, recovery |
| Rec01 | +1% recovery, re-bootstrap | spreads, rates |
| JTD | immediate default: $(1-R)-$ clean value | |

**Why.**
* *Bump the quotes, not the hazard rates.* The market moves spreads. A hedge is another CDS quoted in spread, so risk must be expressed against the same observable. That means re-running the bootstrap inside every bump.
* *CS01 vs RPV01.* To first order the clean value is $(S-c)\times$RPV01, so CS01 $\approx$ RPV01 $\times$ 1bp. They differ because RPV01 itself falls as spreads widen. The primer notes the two terms are often used interchangeably; the table shows by how much they actually differ.
* *Bucketing.* It shows which hedge instrument offsets the risk. A 5Y contract should load almost entirely on the 5Y quote.
* *Rec01 is small.* With spreads fixed, a higher recovery raises the implied hazard and the two effects nearly cancel on the protection leg. What remains comes from the annuity. This is why recovery risk is invisible on a par contract and grows as the trade moves away from par.
* *JTD.* On default the buyer receives $1-R$, pays the accrued coupon and gives up the contract's current value. The accrued and the dirty value combine into the clean value, which leaves the one-line formula.

In [29]:
def clean_value(spreads, recovery, lnP_fn, df_nodes, cds_recovery=None):
    hz = bootstrap(VAL, MATS, spreads, recovery, lnP_fn, df_nodes)
    return value(VAL, T5, 0.01, cds_recovery or recovery, lnP_fn, credit_curve(VAL, MATS, hz), df_nodes + MATS)

BP = 1e-4
base = clean_value(WFC, 0.50, lnP, DF_DATES)
cs01 = clean_value([s + BP for s in WFC], 0.50, lnP, DF_DATES)["pv_clean"] - base["pv_clean"]
buckets = [clean_value([s + BP * (i == j) for j, s in enumerate(WFC)], 0.50, lnP, DF_DATES)["pv_clean"] - base["pv_clean"]
           for i in range(len(WFC))]
lnP_up = lambda d: lnP(d) - BP * act365(VAL, d)          # every zero rate +1bp
ir01 = clean_value(WFC, 0.50, lnP_up, DF_DATES)["pv_clean"] - base["pv_clean"]
rec01 = clean_value(WFC, 0.51, lnP, DF_DATES)["pv_clean"] - base["pv_clean"]
jtd = (1 - 0.50) - base["pv_clean"]

rep = eng.risk_report(eng.CDS(T5, 0.01, 0.50), VAL, MATS, WFC, DISC)
assert np.allclose([cs01, ir01, rec01, jtd, *buckets], [rep.cs01, rep.ir01, rep.rec01, rep.jtd, *rep.cs01_buckets], atol=1e-12)

N = 10_000_000
pd.Series({"clean value": base["pv_clean"] * N, "RPV01 x 1bp": base["rpv01_clean"] * BP * N, "CS01": cs01 * N,
           **{f"CS01 {k}Y bucket": b * N for k, b in zip((1, 2, 3, 5, 7), buckets)},
           "IR01": ir01 * N, "Rec01": rec01 * N, "JTD": jtd * N}, name="USD on 10m notional").to_frame()

,USD on 10m notional
clean value,"112,134.842112"
RPV01 x 1bp,"4,485.393684"
CS01,"4,428.595260"
CS01 1Y bucket,-4.187353
CS01 2Y bucket,-8.546242
CS01 3Y bucket,-19.520461
CS01 5Y bucket,"4,462.121346"
CS01 7Y bucket,0.000000
IR01,-28.406232
Rec01,-112.165708


# Part 11 — Cross-check against QuantLib

**Step.** Put the same curves into QuantLib and price the same contracts with its `IsdaCdsEngine`, then with the `MidPointCdsEngine` used in the course notebook. To isolate the pricing logic, the comparison uses the course notebook's flat 2.95% rate and its 40% recovery spreads.

**Why.** An independent implementation of the same model is the strongest test available without the ISDA C library. Agreement to machine precision means the formulas and date offsets are the same; a small, explained difference means a convention differs; an unexplained one means a bug.

Two details of the set-up matter:
* `Actual360(True)` for the last period reproduces the extra accrual day of 3.4;
* QuantLib observes survival one day before the *payment* date in every period, whereas this engine by default observes one day before the *accrual end*, which in the final period is the maturity date itself. `quantlib_compat=True` switches to QuantLib's choice so the two can be compared exactly.

In [30]:
import QuantLib as ql

def qd(d): return ql.Date(d.day, d.month, d.year)

ql.Settings.instance().evaluationDate = qd(VAL)
RATE, REC = 0.0295, 0.40
NB_SPREADS = [0.0050, 0.0077, 0.0094, 0.01095, 0.0125]
NB_MATS = [eng.cds_maturity(VAL, 12 * k) for k in range(1, 6)]
flat_disc = eng.DiscountCurve.flat(VAL, RATE)
nb_curve = eng.bootstrap_credit_curve(VAL, NB_MATS, NB_SPREADS, REC, flat_disc)

yts = ql.YieldTermStructureHandle(ql.FlatForward(qd(VAL), RATE, ql.Actual365Fixed()))
hc = ql.HazardRateCurve([qd(VAL)] + [qd(m) for m in NB_MATS], [nb_curve.hazards[0]] + nb_curve.hazards, ql.Actual365Fixed())
hc.enableExtrapolation()
prob = ql.DefaultProbabilityTermStructureHandle(hc)
E = ql.IsdaCdsEngine

rows = []
for m, s in zip(NB_MATS, NB_SPREADS):
    sch = ql.Schedule(qd(VAL), qd(m), ql.Period(3, ql.Months), ql.WeekendsOnly(), ql.Following,
                      ql.Unadjusted, ql.DateGeneration.CDS2015, False)
    q = ql.CreditDefaultSwap(ql.Protection.Buyer, 1.0, s, sch, ql.Following, ql.Actual360(), True, True,
                             qd(VAL), ql.FaceValueClaim(), ql.Actual360(True), True, qd(VAL), 3)
    q.setPricingEngine(E(prob, REC, yts, False, E.Taylor, E.NoBias, E.Piecewise))
    isda_prot, isda_prem, isda_fair = q.defaultLegNPV(), -q.couponLegNPV(), q.fairSpread()
    q.setPricingEngine(ql.MidPointCdsEngine(prob, REC, yts))
    mid_prot = q.defaultLegNPV()

    ours = eng.value_cds(eng.CDS(m, s, REC), VAL, flat_disc, nb_curve)
    compat = eng.value_cds(eng.CDS(m, s, REC), VAL, flat_disc, nb_curve, quantlib_compat=True)
    assert abs(compat.protection_pv - isda_prot) < 1e-14 and abs(s * compat.rpv01_dirty - isda_prem) < 1e-14
    rows.append({"maturity": m,
                 "protection: ours - QL ISDA": ours.protection_pv - isda_prot,
                 "premium (compat): ours - QL ISDA": s * compat.rpv01_dirty - isda_prem,
                 "premium (default): ours - QL ISDA": s * ours.rpv01_dirty - isda_prem,
                 "par spread: ours - QL ISDA (bp)": (ours.par_spread - isda_fair) * 1e4,
                 "protection: ours - QL midpoint": ours.protection_pv - mid_prot})
pd.set_option("display.float_format", lambda x: f"{x:.2e}")
pd.DataFrame(rows).set_index("maturity")

,protection: ours - QL ISDA,premium (compat): ours - QL ISDA,premium (default): ours - QL ISDA,par spread: ours - QL ISDA (bp),protection: ours - QL midpoint
maturity,,,,,
2021-12-20,-8.67e-19,-1.86e-16,-1.53e-10,1.52e-06,-1.86e-07
2022-12-20,-1.91e-17,9.02e-17,-4.75e-10,2.43e-06,-4.40e-07
2023-12-20,6.94e-18,-1.39e-16,-6.82e-10,2.38e-06,-4.90e-07
2024-12-20,-2.78e-17,-4.16e-16,-9.24e-10,2.49e-06,-6.78e-07
2025-12-20,2.78e-17,1.80e-16,-1.24e-09,2.75e-06,-7.14e-07


Reading the table, per unit of notional:

* **Protection leg**: identical to QuantLib's ISDA engine to machine precision.
* **Premium leg, compat mode**: identical to machine precision, which confirms the coupon and accrual-on-default formulas.
* **Premium leg, default mode**: differs in the ninth or tenth decimal place, entirely from the final-period observation date. On USD 10m that is about one cent at most.
* **Par spread**: agrees to within a few millionths of a basis point.
* **Midpoint engine**: differs in the seventh decimal. This is the approximation error measured in Part 5.

One caveat on scope: this validates the engine against QuantLib's implementation of the ISDA model, not against the ISDA C library or a Markit calculator screen. The observation-date difference above is exactly the kind of detail that only the reference code can settle; the default here follows the convention that the final period accrues through maturity.

# Summary of design choices

| Choice | Reason |
|---|---|
| Log-linear discount factors | Equivalent to constant forward rates; positive, monotone, and the ISDA standard |
| Piecewise-constant hazard rate | One unknown per quote, so the bootstrap is exactly determined |
| Segments cut at every node of both curves | Makes $DF\cdot Q$ a single exponential on each segment, hence closed-form integrals |
| Taylor series for small $\hat f+\hat h$ | Avoids $0/0$ in the closed forms |
| Act/360 for coupons, Act/365F for curve time | The first is contractual, the second a modelling clock |
| Step-in at $T+1$, full first coupon, accrued rebate | Makes all contracts on a name fungible |
| Accrual on default integrated exactly | The half-coupon shortcut fails for wide spreads |
| Root-finding bootstrap on the clean value | The ISDA legs are non-linear in the hazard rate; par means clean value zero |
| Hard errors for impossible quotes | A negative survival probability is a data problem, not a result |
| Conventional spread via a flat curve | Both counterparties must derive the same cash amount from one number |
| Risk by bumping quotes and re-bootstrapping | Risk has to be expressed in the instruments used to hedge it |

**Not covered in this version:** building the discount curve from deposit and swap quotes (discount factors are taken as given), holiday calendars beyond weekends, index and tranche products, and validation against the ISDA C library.